# Chapter 5: The First Learning Neuron, Live Python Exercises

**ECUE 3.1.3: Neural Communication** &bull; CESI Bachelor AI (Year 2)

Exercises for **Chapter 5, The First Learning Neuron**: the perceptron learning rule, the training loop on AND, OR and NOT, new gates learned without picking any weight, the many solutions of one problem, a probe on XOR, a first decoder of firing rates, and two exercises to go further: a mislabelled trial, and a decoder that reads spike trains.

Each exercise has a **`# TODO`** cell for you to fill in during the live session. Reference solutions are in `../notebooks-instructor/Chapter5_LiveExercises_SOLUTIONS.ipynb`: try first, then compare.

<div style="border-left: 4px solid #f59e0b; padding: 0.5em 1em; background: rgba(245,158,11,0.08);">
<b>Sanity check:</b> run the cell below. If it prints a NumPy version with no error, your environment is ready. If not, go back to <code>Chapter1_Setup.ipynb</code>.
</div>

In [ ]:
import numpy as np

print("NumPy version:", np.__version__)
print("Environment OK ✔")

---

## Reminder: the McCulloch-Pitts Neuron

The perceptron keeps **exactly the same neuron** as Chapter 4. Run the cell below so `mp_neuron` is available in this notebook.

In [ ]:
def mp_neuron(x, w, b):
    z = np.dot(w, x) + b       # weighted sum + bias (spatial summation)
    return 1 if z >= 0 else 0  # step function (all-or-nothing decision)

---

## Live Exercise: One Learning Step

Recall the perceptron learning rule:

$$w_i \leftarrow w_i + \eta\,(t - y)\,x_i \qquad\qquad b \leftarrow b + \eta\,(t - y)$$

where $e = t - y$ is the **error** (in the code: `e`) and $\eta$ is the **learning rate** (in the code: `eta`).

### Your turn

Write `perceptron_update(x, t, w, b, eta)` that:
1. computes the neuron's output `y` with `mp_neuron`,
2. computes the error `e = t - y`, then updates `w` and `b` with the rule above,
3. returns the new `w`, the new `b`, and `y` (the answer **before** the update).

In [ ]:
def perceptron_update(x, t, w, b, eta=1.0):
    # TODO: compute y, then the error e = t - y, then apply the perceptron rule to w and b
    ...
    return w, b, y

### Check it against your paper table

The cell below runs **epoch 1 on OR**, from $w_1 = w_2 = 0$, $b = 0$, $\eta = 1$, and prints one line per row. Compare every line with the table you filled in by hand.

In [ ]:
X_or = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
T_or = np.array([0, 1, 1, 1])

w, b = np.zeros(2), 0.0
print(" x1 x2 | t | old w1 old w2 old b |  z | y |  e | new w1 new w2 new b")
print("-" * 70)
for x, t in zip(X_or, T_or):
    old_w, old_b = w, b
    z = np.dot(old_w, x) + old_b
    w, b, y = perceptron_update(x, t, w, b, eta=1.0)
    print(f"  {x[0]}  {x[1]} | {t} | {old_w[0]:6.1f} {old_w[1]:6.1f} {old_b:5.1f} | {z:2.0f} | {y} | {t - y:2d} | "
          f"{w[0]:6.1f} {w[1]:6.1f} {b:5.1f}")

---

## Live Exercise: The Training Loop

```text
initialize w = 0, b = 0
repeat for each epoch:
    for each example (x, t):
        apply the perceptron rule
    if no mistakes in this epoch:
        stop
```

### Your turn

Write `train_perceptron(X, T, eta, n_epochs)` that runs the loop above, using your `perceptron_update`. It must return the learned `w` and `b`, and `mistakes`: a list with the **number of mistakes in each epoch**.

In [ ]:
def train_perceptron(X, T, eta=1.0, n_epochs=20):
    w, b = np.zeros(X.shape[1]), 0.0
    mistakes = []
    # TODO: loop over epochs, then over examples; count mistakes; stop early on a perfect epoch
    ...
    return w, b, mistakes

### Training OR, AND and NOT

Compare the learned weights with the ones you picked by hand in Chapter 4: AND $(1, 1, -1.5)$, OR $(1, 1, -0.5)$, NOT $(-1, 0.5)$.

**Before running**: do you expect the neuron to learn *exactly* these values?

In [ ]:
X_2 = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
X_1 = np.array([[0], [1]])

gates = {
    "OR":  (X_2, np.array([0, 1, 1, 1])),
    "AND": (X_2, np.array([0, 0, 0, 1])),
    "NOT": (X_1, np.array([1, 0])),
}

for name, (X, T) in gates.items():
    w, b, mistakes = train_perceptron(X, T)
    outputs = [mp_neuron(x, w, b) for x in X]
    print(f"{name:3s} | w = {w}, b = {b} | mistakes per epoch: {mistakes} | outputs: {outputs}, targets: {T.tolist()}")

---

## Live Exercise: New Gates, Nobody Picks the Weights

In Chapter 4, **you** picked the weights of every gate. Now the rule does it. Three gates you have not trained yet:

| Gate | Inputs | Output is 1 when... |
|:---:|:---:|:---|
| NAND | 2 | NOT (x1 AND x2): every case except (1, 1) |
| NOR | 2 | NOT (x1 OR x2): only (0, 0) |
| MAJORITY | 3 | at least 2 of the 3 inputs are 1 |

Careful: **NOR is not NOT**. NOT flips **one** input; NOR first combines **two** inputs with OR, then flips the result. Feed NOR the same input twice, $x_1 = x_2 = x$, and it behaves exactly like NOT.

### Your turn

1. **On paper first**: find weights and a bias for MAJORITY by hand.
2. Write the targets of the three gates, train them with `train_perceptron`, and check the outputs.
3. Compare the learned MAJORITY neuron with yours. Does it treat the three inputs the same way? Why does that make sense?
4. Compare the learned NOR with the NOT learned in the previous exercise. What do you notice?

In [ ]:
X_3 = np.array([[a, b, c] for a in (0, 1) for b in (0, 1) for c in (0, 1)])   # the 8 combinations of 3 inputs
print(X_3)

new_gates = {
    "NAND":     (X_2, ...),   # TODO: the 4 targets of NAND
    "NOR":      (X_2, ...),   # TODO: the 4 targets of NOR
    "MAJORITY": (X_3, ...),   # TODO: the 8 targets of MAJORITY
}

# TODO: train each gate with train_perceptron, and print w, b, the mistakes per epoch, the outputs and the targets

---

## Think: Can One Neuron Build Every Gate?

In Chapter 4, no MCP neuron could reproduce XOR. What about the new gates? **Discuss first, then check.**

1. Can a single MCP neuron reproduce NAND, NOR and MAJORITY? How do you know, **without** searching for weights?
2. With 2 inputs, a truth table has 4 rows, so there are $2^4 = 16$ possible gates. How many can **one** neuron reproduce?
3. Which ones are **impossible**, like XOR?

### Check it

Reuse the grid search of Chapter 4: for each of the 16 truth tables below, look for a triple $(w_1, w_2, b)$ on a grid that reproduces it. The cell prints the full matrix: one line per gate, its 4 outputs, and whether one neuron can build it.

In [ ]:
grid = np.arange(-2, 2.01, 0.5)

# the 16 gates of two inputs: targets for (0, 0), (0, 1), (1, 0), (1, 1)
gates_16 = {
    "FALSE (always 0)":   [0, 0, 0, 0],
    "AND":                [0, 0, 0, 1],
    "x1 AND NOT x2":      [0, 0, 1, 0],
    "x1 (copy)":          [0, 0, 1, 1],
    "NOT x1 AND x2":      [0, 1, 0, 0],
    "x2 (copy)":          [0, 1, 0, 1],
    "XOR":                [0, 1, 1, 0],
    "OR":                 [0, 1, 1, 1],
    "NOR":                [1, 0, 0, 0],
    "XNOR":               [1, 0, 0, 1],
    "NOT x2":             [1, 0, 1, 0],
    "x1 OR NOT x2":       [1, 0, 1, 1],
    "NOT x1":             [1, 1, 0, 0],
    "NOT x1 OR x2":       [1, 1, 0, 1],
    "NAND":               [1, 1, 1, 0],
    "TRUE (always 1)":    [1, 1, 1, 1],
}

print(f"{'gate':18s} | (0,0) (0,1) (1,0) (1,1) | 1 neuron?")
print("-" * 54)
n_possible = 0
for name, T in gates_16.items():
    found = False
    # TODO: search the grid for (w1, w2, b) such that [mp_neuron(x, [w1, w2], b) for x in X_2] == T
    ...
    n_possible += int(found)
    print(f"{name:18s} |   {T[0]}     {T[1]}     {T[2]}     {T[3]}   | {'yes' if found else 'NO'}")

print(n_possible, "gates out of 16 fit in one neuron")

---

## Live Exercise: The Perceptron as a Class

Until now, the state of the neuron (`w`, `b`, `mistakes`) travels between functions as separate variables. A **class** keeps the state and the functions that use it together, in one object:

- **attributes** (`self.w`, `self.b`, `self.mistakes_`): the state of *this* neuron, kept from one call to the next;
- **methods** (`predict`, `fit`): functions that read or change that state. `self` is the neuron itself.

`__init__` runs once, when the object is created: `p = Perceptron(n_features=2)`. Then `p.fit(X, T)` trains it, `p.predict(X)` decodes, and `p.w` reads its weights. This is the shape of the decoder you will write in Project 1.

Two upgrades on the way:

1. **Predict all the examples at once.** With one example per row of `X`, `X @ self.w + self.b` computes the weighted sum of **every row**: a vector of $z$, one per example. `(z >= 0).astype(int)` applies the step function to all of them.
2. **A random order.** `fit` visits the examples in a new random order at each epoch: `self.rng.permutation(len(X))` returns the indices `0, 1, ..., n - 1`, shuffled.

### Your turn

Complete `predict` and `fit`. `fit` does exactly what `train_perceptron` does (reuse `perceptron_update`), except the order of the examples. It returns `self`, so that you can write `Perceptron(2).fit(X, T)` in one line.

In [ ]:
class Perceptron:
    def __init__(self, n_features, eta=1.0, epochs=20, seed=0):
        self.w = np.zeros(n_features)
        self.b = 0.0
        self.eta = eta
        self.epochs = epochs
        self.rng = np.random.default_rng(seed)
        self.mistakes_ = []  # number of mistakes per epoch

    def predict(self, X):
        # TODO: return an array of 0/1 predictions, one per row of X
        ...

    def fit(self, X, T):
        # TODO: the loop of train_perceptron, visiting the examples in a new random order at each epoch
        ...
        return self

### Check it on OR, AND and NOT

In [ ]:
for name, (X, T) in gates.items():
    p = Perceptron(n_features=X.shape[1]).fit(X, T)
    print(f"{name:3s} | w = {p.w}, b = {p.b} | mistakes per epoch: {p.mistakes_} | "
          f"outputs: {p.predict(X).tolist()}, targets: {T.tolist()}")

**Question**: compare with `train_perceptron` above. Are the outputs the same? The weights? The number of epochs? Why is visiting the examples in a random order a good habit?

---

## Live Exercise: One Problem, Many Solutions

The `seed` changes the order in which `fit` visits the examples. Does it change the answer?

### Your turn

1. Train a `Perceptron` on AND with each seed from 0 to 9. Print the learned `w`, `b`, and the number of epochs.
2. Keep each **different** solution in a list `solutions`, as a tuple `(w1, w2, b)`. How many are there? Do they all compute AND?
3. For each solution, print $z$ on the 4 inputs, and do the same with your hand-picked weights from Chapter 4, $(1, 1)$ and $-1.5$. Imagine a noisy input, `[0.9, 1]` instead of `[1, 1]`: which neuron would you trust?

In [ ]:
T_and = np.array([0, 0, 0, 1])
solutions = []

# TODO: for seed in range(10): train Perceptron(2, seed=seed) on AND, print it, and add (w1, w2, b) to solutions if it is new

In [ ]:
# TODO: for each solution, and for the hand-picked (1, 1, -1.5): print z on the 4 rows of X_2

---

## And XOR?

The convergence theorem says the rule finds a solution **if one exists**. Run the cell below and look at the mistakes per epoch.

In [ ]:
T_xor = np.array([0, 1, 1, 0])

w, b, mistakes = train_perceptron(X_2, T_xor, n_epochs=20)
print("mistakes per epoch:", mistakes)
print("final w =", w, ", b =", b)
print("outputs:", [mp_neuron(x, w, b) for x in X_2], "targets:", T_xor.tolist())

**Question**: does increasing `n_epochs` to 1000 help? Try it. Why do you think the rule never stops making mistakes? *(We answer this properly in Chapter 6.)*

In [ ]:
# Try n_epochs=1000 here

---

## Chapter Bilan: Decode a Stimulus From Firing Rates

*Synthesis exercise, and a first taste of Project 1. No scaffolding this time.*

Three neurons were recorded over 8 trials. On each trial, one of two stimuli was shown: **A** (target `0`) or **B** (target `1`). Each input is a firing rate in Hz.

| Trial | $x_1$ *(Hz)* | $x_2$ *(Hz)* | $x_3$ *(Hz)* | Stimulus |
|:---:|:---:|:---:|:---:|:---:|
| 1 | 12 | 35 | 20 | A *(0)* |
| 2 | 8  | 40 | 25 | A *(0)* |
| 3 | 15 | 30 | 18 | A *(0)* |
| 4 | 10 | 38 | 22 | A *(0)* |
| 5 | 30 | 15 | 21 | B *(1)* |
| 6 | 35 | 10 | 19 | B *(1)* |
| 7 | 28 | 18 | 24 | B *(1)* |
| 8 | 33 | 12 | 20 | B *(1)* |

### Your turn

1. Train `train_perceptron` on these 8 trials.
2. Predict the stimulus for the 4 **new** trials in `X_new`, which the neuron has never seen.
3. **Evaluate** the decoder on these new trials: accuracy and confusion matrix.
4. Plot **two figures**: the mistakes per epoch, and the learned weights.
5. Look at the learned `w`: which neuron **matters most**, and which one **carries no information** about the stimulus?

In [ ]:
X_train = np.array([[12, 35, 20], [8, 40, 25], [15, 30, 18], [10, 38, 22],
                    [30, 15, 21], [35, 10, 19], [28, 18, 24], [33, 12, 20]])
T_train = np.array([0, 0, 0, 0, 1, 1, 1, 1])

X_new = np.array([[11, 33, 23], [31, 14, 22], [14, 36, 19], [26, 20, 18]])
T_new = np.array([0, 1, 0, 1])   # true stimuli of the new trials: A, B, A, B

In [ ]:
# Your turn: train, then decode X_new

---

### Step 3: evaluate the decoder

The true stimuli of the 4 new trials are `T_new = [0, 1, 0, 1]`. Compute the **accuracy** on these test trials, and the **confusion matrix** (rows: true stimulus, columns: prediction). This is how both projects measure a decoder: always on trials it has never seen.

In [ ]:
# Step 3: accuracy and confusion matrix on the new trials

### Step 4: two figures

A result is easier to read, and to defend in a report, with a figure. Plot side by side:
1. the **mistakes per epoch** (`mistakes`, returned by `train_perceptron`): does the training converge?
2. the **learned weights**, one bar per neuron (`plt.bar`): which ones are large, which ones are close to 0?

*Hint*: `fig, axes = plt.subplots(1, 2)`, as in Chapter 7, gives two plots side by side.

In [ ]:
# Step 4: mistakes per epoch, and one bar per learned weight

### Step 5: read the weights

Which neuron **matters most**, and which one **carries no information** about the stimulus? Compare the learned `w` with the table of firing rates.

In [ ]:
# Step 5: print w, and compare it with the table of firing rates

---

### Bonus 1 (optional): the learning rate

Retrain the decoder with `eta=0.1` instead of `1.0`. Compare the learned `w`, `b`, the number of epochs, and the predictions. What changed, and what did not?

In [ ]:
# Bonus 1: retrain with eta=0.1 and compare

### Bonus 2 (optional): standardize the features

Real recordings need standardized features (see the slides). Compute the mean `mu` and standard deviation `sigma` of each neuron **on the training trials**, standardize `X_train` and `X_new` with them, retrain, and decode again.

In [ ]:
# Bonus 2: z-score each neuron with the training statistics, retrain, decode

---

### Bonus 3 (optional): when the learning rate does matter

Bonus 1 showed that, **starting from zero**, $\eta$ only rescales the weights. Now start from a neuron that already has an opinion, and a wrong one: $w = (-1, 1, 0)$, $b = 0$, a neuron that prefers stimulus A.

1. Write `train_from(X, T, w0, b0, eta, n_epochs)`: a copy of `train_perceptron` that starts from `w0` and `b0` instead of zeros.
2. Train the decoder from $(-1, 1, 0)$ with `eta = 1`, `0.1` and `0.01`. Compare the learned weights, the number of epochs, and the predictions on `X_new`.

In [ ]:
def train_from(X, T, w0, b0, eta=1.0, n_epochs=100):
    w, b = np.array(w0, dtype=float), float(b0)
    mistakes = []
    # TODO: the same loop as train_perceptron
    ...
    return w, b, mistakes

# TODO: train from w0 = [-1, 1, 0], b0 = 0 with eta = 1, 0.1 and 0.01, and decode X_new

---

## Going Further: One Wrong Label

On a real experiment, a trial is sometimes **mislabelled**: the wrong stimulus is written in the log. Here, trial 4 was an A, but it was logged as B.

### Your turn

1. Train `train_perceptron` on `T_noisy` for 50 epochs. Look at the mistakes per epoch: does the training stop? Decode `X_new` and compute the test accuracy.
2. Write `train_pocket(X, T, eta, n_epochs)`. It runs the same loop for all the epochs, but at the **end of each epoch** it counts the mistakes of the current `w`, `b` on the **whole** training set, and keeps in its "pocket" the weights with the fewest mistakes seen so far. It returns these best weights, and the list of mistakes per epoch.
3. Decode `X_new` with the pocket weights. Better?

In [ ]:
T_noisy = T_train.copy()
T_noisy[3] = 1          # trial 4 was an A, logged as B
print("T_noisy:", T_noisy)

# TODO 1: train_perceptron for 50 epochs on T_noisy, then decode X_new

In [ ]:
def train_pocket(X, T, eta=1.0, n_epochs=50):
    w, b = np.zeros(X.shape[1]), 0.0
    best_w, best_b, best_errors = w, b, len(X) + 1
    history = []
    # TODO: for each epoch, apply the rule on every example,
    #       then count the mistakes of (w, b) on the whole training set,
    #       and keep (w, b) in the pocket if they beat best_errors
    ...
    return best_w, best_b, history

# TODO 3: decode X_new with the pocket weights

---

## Going Further: From Spike Trains to a Decoder

Until now, the firing rates were given. In a real experiment, you record **spikes** and **count** them, as in the Chapter 3 warm-up. Five neurons are recorded, and two stimuli are shown:

| Neuron | 1 | 2 | 3 | 4 | 5 |
|:---:|:---:|:---:|:---:|:---:|:---:|
| Mean rate for A *(Hz)* | 20 | 35 | 15 | 30 | 25 |
| Mean rate for B *(Hz)* | 30 | 25 | 15 | 30 | 25 |

The cell below simulates the recordings: `record` draws spikes in 1 ms bins, like `spike_train` in Chapter 3, and returns the firing rate of each neuron on each trial. `dataset` returns 20 training trials and 10 test trials per stimulus.

### Your turn

1. **Before coding**: which neurons should the decoder use, and which should it ignore?
2. Generate a dataset with trials of 1 s. Standardize it with the **training** statistics, train a `Perceptron(5, epochs=50)`, and compute the test accuracy and the confusion matrix.
3. Plot the learned weights, one bar per neuron. Does the decoder agree with your answer to question 1?
4. **How long must the decoder listen?** For trials of 0.05, 0.1, 0.2, 0.5, 1 and 2 s, redo step 2 five times (five new datasets) and average the test accuracy. Plot the accuracy against the duration.

In [ ]:
rng = np.random.default_rng(2026)
dt = 0.001                                  # 1 ms bins

rates_A = np.array([20, 35, 15, 30, 25])    # Hz, stimulus A
rates_B = np.array([30, 25, 15, 30, 25])    # Hz, stimulus B

def record(rates, n_trials, duration, rng):
    n_bins = int(duration / dt)
    X = np.zeros((n_trials, len(rates)))
    for k in range(n_trials):
        spikes = (rng.random((len(rates), n_bins)) < rates[:, None] * dt).astype(int)
        X[k] = spikes.sum(axis=1) / duration    # count / duration = firing rate in Hz
    return X

def dataset(duration, rng):
    X_tr = np.vstack([record(rates_A, 20, duration, rng), record(rates_B, 20, duration, rng)])
    T_tr = np.array([0] * 20 + [1] * 20)
    X_te = np.vstack([record(rates_A, 10, duration, rng), record(rates_B, 10, duration, rng)])
    T_te = np.array([0] * 10 + [1] * 10)
    return X_tr, T_tr, X_te, T_te

X_tr, T_tr, X_te, T_te = dataset(1.0, rng)
print("first 3 training trials (Hz):")
print(X_tr[:3])

In [ ]:
# TODO 2: standardize with the training statistics, train Perceptron(5, epochs=50), test accuracy and confusion matrix

In [ ]:
# TODO 3: one bar per learned weight

In [ ]:
durations = [0.05, 0.1, 0.2, 0.5, 1.0, 2.0]
# TODO 4: for each duration, the mean test accuracy over 5 new datasets; then plot accuracy against duration